# 00 Orchestrator


In [ ]:
%pip install -q google-cloud-bigquery pandas pyarrow db-dtypes requests papermill
import os,sys,json,pandas as pd
from google.cloud import bigquery
PROJECT_ID="your-gcp-project-id"  # замініть у всіх ноутбуках
LOCATION="EU"; creds=None
if "google.colab" in sys.modules:
 from google.colab import auth; auth.authenticate_user()
elif os.environ.get("GCP_SA_KEY"):
 from google.oauth2 import service_account
 creds=service_account.Credentials.from_service_account_info(json.loads(os.environ["GCP_SA_KEY"]),scopes=["https://www.googleapis.com/auth/cloud-platform"])
client=bigquery.Client(project=PROJECT_ID,location=LOCATION,credentials=creds)
print("проєкт:",client.project)

In [ ]:
import uuid,papermill as pm
from pathlib import Path
N=["01_bronze.ipynb","02_dim_currency.ipynb","03_dim_date.ipynb","04_snapshot.ipynb","05_fact_merge.ipynb"]; Path("runs").mkdir(exist_ok=True); rid=str(uuid.uuid4()); logs=[]
for i,name in enumerate(N,1):
 a=pd.Timestamp.now(tz="UTC")
 try: pm.execute_notebook(name,f"runs/{name}",kernel_name="python3"); st,err="OK",None
 except Exception as e: st,err="FAILED",f"{type(e).__name__}: {e}"[:1000]
 z=pd.Timestamp.now(tz="UTC"); logs.append({"run_id":rid,"step_no":i,"notebook":name,"status":st,"started_at":a,"finished_at":z,"duration_sec":(z-a).total_seconds(),"error":err}); print(name,st)
 if st=="FAILED": print("Stopped at",name); break
L=pd.DataFrame(logs); M=f"{PROJECT_ID}.nbu_meta"; T=f"{M}.run_log"; ds=bigquery.Dataset(M); ds.location=LOCATION; client.create_dataset(ds,exists_ok=True); s=[bigquery.SchemaField("run_id","STRING"),bigquery.SchemaField("step_no","INTEGER"),bigquery.SchemaField("notebook","STRING"),bigquery.SchemaField("status","STRING"),bigquery.SchemaField("started_at","TIMESTAMP"),bigquery.SchemaField("finished_at","TIMESTAMP"),bigquery.SchemaField("duration_sec","FLOAT"),bigquery.SchemaField("error","STRING")]; t=bigquery.Table(T,schema=s); t.time_partitioning=bigquery.TimePartitioning(field="started_at"); client.create_table(t,exists_ok=True); client.load_table_from_dataframe(L,T,job_config=bigquery.LoadJobConfig(schema=s,write_disposition="WRITE_APPEND")).result(); print("OK",(L.status=="OK").sum(),"FAILED",(L.status=="FAILED").sum(),"seconds",round(L.duration_sec.sum(),2)); display(client.query(f"SELECT run_id,step_no,notebook,status,duration_sec FROM `{T}` ORDER BY started_at DESC LIMIT 10").to_dataframe())